# Lesson 3 — Three Artifact Checks, One Quality Gate

**Required · real-recording replay · about 60 minutes**

Lesson 2 answered: *Can we read the file?* This lesson asks:
*Which channels and one-second windows need inspection before analysis?*

**Real EEG → original NeuraDock detectors → three result plots → quality gate.**

We use NeuraDock's original `eeg_quality_check()` and `clean_eeg_data()`
functions, not a new algorithm invented for this tutorial. A small safety
wrapper handles two boundary cases and preserves the original sample timeline.

By the end, you can explain the three metrics, read a flagged-window plot,
and distinguish **screening**, **excluding data**, and **repairing data**.
You will generate six figures from a real file; no artifacts are injected.

**VS Code:** select `neuradock_env` (or your course Python environment), then
**Run All**. Run unchanged first; afterwards change `CHANNEL_INDEX` or
`VIEW_START_S` and rerun from the settings cell. A real file is required.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

locations = (Path.cwd(), Path.cwd() / "neuradock-eeg-101", *Path.cwd().parents)
ROOT = next((p for p in locations if (p / "src" / "neuradock_eeg101").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the neuradock-eeg-101 project folder in VS Code first.")
sys.path.insert(0, str(ROOT / "src"))
IN_NOTEBOOK = "ipykernel" in sys.modules
if IN_NOTEBOOK:
    get_ipython().run_line_magic("matplotlib", "inline")
else:
    os.environ.setdefault("MPLBACKEND", "Agg")
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.signal import butter, filtfilt, welch
from typing import Tuple, Dict
from neuradock_eeg101.reading import data_reader, inspect_timing, describe_p_field
from neuradock_eeg101.vendor.quality_tools import PROFILE
from neuradock_eeg101.quality_gate import gate_from_metrics

OUTPUT_DIR = ROOT / "outputs" / "notebooks" / "lesson-03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES = []
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})

def show_and_save(fig, filename):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight", facecolor="white")
    FIGURES.append(filename)
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)


## 1. Keep the real input and its units · 5 min

This uses the same reader and default real recording as Lesson 2. No values
are rescaled. On 2026-09-21, the data owner confirmed that the supplied
`S04_01.txt` and `S04_11.txt` values are already **microvolts (µV)**.
That confirmation is linked to their file hashes, not merely their names.

For a different file, set `DATA_PATH` and confirm its acquisition settings.
Set `UNIT_CONFIRMED = True` **only after verifying that its values are µV**.
With unknown units, this notebook shows reference metrics, but does not
authorize a quality gate or report an accepted duration.

Channel labels remain `Ch1`–`Ch7`: a seven-column file does not establish
its historical electrode montage. The 250 Hz rate is declared in the S2
acquisition notebook. Relative time is sample index / 250, not repaired
device-clock timing. Private recordings and executed outputs stay out of Git.


In [ ]:
TEACHING_FILE = ROOT / "data" / "teaching" / "lesson-02" / "recording-01.txt"
LOCAL_ORIGINAL = ROOT.parent / "S2" / "S04_01.txt"
default_file = TEACHING_FILE if TEACHING_FILE.is_file() else LOCAL_ORIGINAL
DATA_PATH = Path(os.environ.get("NEURADOCK_LESSON03_FILE",
                 os.environ.get("NEURADOCK_LESSON02_FILE", str(default_file))))
if not DATA_PATH.is_absolute():
    DATA_PATH = ROOT / DATA_PATH
DATA_PATH = DATA_PATH.resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Lesson 3 requires a real recording. Set DATA_PATH or "
        "NEURADOCK_LESSON03_FILE to your authorized file. No synthetic substitute will be used.")
if (ROOT / "data" / "synthetic").resolve() in DATA_PATH.parents:
    raise ValueError("Lesson 3 requires real data, not a bundled synthetic recording.")
SOURCE_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
CONFIRMED_UV_HASHES = {
    "1c7cf69cf4572aa96b3452f2febb176c8c2a024eaf0e95812b22edda881006bb",
    "23a8e68b331a86ed7eb79056b580a972bdb9ee74d189e881316f0c7ead35db54",
}
UNIT_CONFIRMED = None  # None = use the two owner-confirmed hashes; otherwise True/False.
unit_confirmed = (SOURCE_SHA256 in CONFIRMED_UV_HASHES
                  if UNIT_CONFIRMED is None else UNIT_CONFIRMED)
if not isinstance(unit_confirmed, bool):
    raise ValueError("UNIT_CONFIRMED must be None, True, or False.")
UNIT = "µV" if unit_confirmed else "recorded units"
REFERENCE_ONLY = "" if unit_confirmed else " [REFERENCE ONLY: units unconfirmed]"
if os.environ.get("NEURADOCK_LESSON03_VERIFY") == "1":
    OUTPUT_DIR = OUTPUT_DIR / "verification" / SOURCE_SHA256[:12]
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 250  # Acquisition setting; the pinned detector requires this course profile.
CHANNEL_NAMES = [f"Ch{i + 1}" for i in range(7)]
CHANNEL_INDEX = 0  # Viewing control: 0 = Ch1, ..., 6 = Ch7.
VIEW_START_S = 0.0  # Viewing control for the first waveform figure only.
THRESHOLDS = (10.0, 20.0, 2.0)  # Original NeuraDock metric thresholds; do not tune for a pass.
print("Input:", DATA_PATH.name, "| Mode: real-recording replay")
print("Units:", UNIT, "| Confirmed:", unit_confirmed)
print("Column labels only; electrode montage remains unconfirmed.")


In [ ]:
data, info = data_reader(DATA_PATH)
n_channels, n_samples = data.shape
if n_channels != 7 or n_samples < FS or not np.isfinite(data).all():
    raise ValueError("Expected seven finite channels and at least one full second.")
if not isinstance(CHANNEL_INDEX, int) or not 0 <= CHANNEL_INDEX < n_channels:
    raise ValueError("CHANNEL_INDEX must be an integer from 0 to 6.")
if not np.isfinite(VIEW_START_S) or not 0 <= VIEW_START_S < n_samples / FS:
    raise ValueError("VIEW_START_S must be inside the recording.")
time_s = np.arange(n_samples) / FS
n_seconds, tail_samples = divmod(n_samples, FS)
seconds = np.arange(n_seconds)
timing = inspect_timing(info["raw_timestamps"], timestamp_format="clock_hms_ms")
p_summary = describe_p_field(info["p_fields"])
print("Shape:", data.shape, "| Nominal duration:", n_samples / FS, "s")
print("Full one-second windows:", n_seconds, "| Unassessed trailing samples:", tail_samples)
print("Malformed rows:", len(info["malformed_rows"]))
print("Repeated/backward packet-clock intervals:", timing["nonpositive_interval_count"])
print("Maximum packet-clock interval:", timing["maximum_packet_interval_s"], "s")
print("The nominal plot axis does not repair clock irregularities or infer events from P.")


## 2. Read the original detector · 10 min

The function below is copied unchanged from NeuraDock's
[`quality_tools.py`](https://github.com/Neuradock/eeg-workstation-agent/blob/e3539cbeca99e824dfeb3ebdb44d80ca80f0d5dc/src/neuradock_agent/quality_tools.py),
pinned for reproducibility. Copyright (c) 2026 NeuraDock; MIT License.
The complete license is retained under `src/neuradock_eeg101/vendor/`.

Read it in this order:

1. Apply the original **4th-order Butterworth 1–50 Hz bandpass** with
   `filtfilt` (offline, zero-phase). There is **no extra notch filter**.
2. Split the filtered recording into **non-overlapping 1-second windows**.
3. Compute a **49–51 Hz PSD-bin sum**, a **20–40 Hz PSD-bin sum**, and
   a **count of samples whose absolute value is at least 100 µV**.

Welch uses 250 samples here, giving a 1 Hz frequency grid. The code sums PSD
bins; it does **not** numerically integrate band power. Keep these definitions
and their thresholds together. Filtering/PSD theory is developed in Lesson 4.
A filter changes the signal used for screening; it does not prove artifacts
were removed. Use one continuous recording, never concatenated unrelated trials.


In [ ]:
def eeg_quality_check(
    eeg_data: np.ndarray,
    fs: int = PROFILE.sampling_rate_hz,
) -> Tuple[Tuple[np.ndarray, np.ndarray, np.ndarray], np.ndarray]:
    """Filter EEG and compute 50 Hz, EMG-band, and outlier metrics per second."""

    matrix = np.asarray(eeg_data, dtype=float)
    if matrix.ndim != 2:
        raise ValueError("eeg_quality_check expects shape (channels, samples).")
    if matrix.shape[1] < fs:
        raise ValueError("Signal quality workflow requires at least one full second.")

    nyquist = 0.5 * fs
    b_filter, a_filter = butter(
        4, [1.0 / nyquist, 50.0 / nyquist], btype="band"
    )
    filtered = filtfilt(b_filter, a_filter, matrix, axis=1)

    segment_length = fs
    n_segments = matrix.shape[1] // segment_length
    metrics = [
        np.zeros((matrix.shape[0], n_segments), dtype=float) for _ in range(3)
    ]
    for channel_index in range(matrix.shape[0]):
        for segment_index in range(n_segments):
            start = segment_index * segment_length
            segment = filtered[channel_index, start : start + segment_length]
            frequencies, psd = welch(
                segment, fs=fs, nperseg=min(len(segment), fs * 2)
            )
            metrics[0][channel_index, segment_index] = np.sum(
                psd[(frequencies >= 49.0) & (frequencies <= 51.0)]
            )
            metrics[1][channel_index, segment_index] = np.sum(
                psd[(frequencies >= 20.0) & (frequencies <= 40.0)]
            )
            metrics[2][channel_index, segment_index] = np.sum(
                (segment <= -PROFILE.quality.outlier_absolute_amplitude)
                | (segment >= PROFILE.quality.outlier_absolute_amplitude)
            )
    return (metrics[0], metrics[1], metrics[2]), filtered


In [ ]:
metrics, filtered = eeg_quality_check(data, fs=FS)
line_metric, high_frequency_metric, outlier_count = metrics
assert filtered.shape == data.shape
assert all(metric.shape == (7, n_seconds) for metric in metrics)
print("Metric array shape:", line_metric.shape, "= channels x full seconds")
print("Raw data is unchanged; filtered is a separate array.")


In [ ]:
first = int(VIEW_START_S * FS)
last = min(first + 5 * FS, n_samples)
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, values, title, color in zip(
        axes, (data, filtered), ("Raw recording", "Detector input: 1–50 Hz bandpass"),
        ("#64748b", "#2563eb")):
    ax.plot(time_s[first:last], values[CHANNEL_INDEX, first:last], color=color, lw=0.9)
    for boundary in range(int(time_s[first]) + 1, int(time_s[last - 1]) + 1):
        ax.axvline(boundary, color="#cbd5e1", ls=":", lw=0.8)
    ax.set(ylabel=f"Amplitude ({UNIT})", title=title)
axes[-1].set_xlabel("Nominal time (s; sample index / 250)")
fig.suptitle(f"1. {CHANNEL_NAMES[CHANNEL_INDEX]}: keep raw and detector input distinct" + REFERENCE_ONLY)
show_and_save(fig, "01-raw-and-filtered.png")


## 3. Three checks, three visible results · 20 min

### A. Power-line-frequency screening

**Rule:** sum the filtered PSD bins at 49, 50, and 51 Hz; flag a window if
the result is **greater than 10**. This detector targets 50 Hz, not 60 Hz.

The left panel compares raw and filtered spectra for the selected channel's
**largest line-metric window** (an explicitly chosen inspection example).
The right panel shows **every full second**, the threshold, and flags.
The bandpass attenuates around 50 Hz, so this score is not the original
recording's unfiltered line-noise strength. A flag suggests inspection, not
a proven electrical source; no flags is also a legitimate result.
The score axis is linear below the threshold and logarithmic above it, so
one very large transient does not hide all the ordinary windows. Scores
themselves are unchanged.


In [ ]:
ch = CHANNEL_INDEX
example_second = int(np.argmax(line_metric[ch]))
window = slice(example_second * FS, (example_second + 1) * FS)
freq, raw_psd = welch(data[ch, window], fs=FS, nperseg=FS)
_, detector_psd = welch(filtered[ch, window], fs=FS, nperseg=FS)
line_bins = (freq >= 49.0) & (freq <= 51.0)
# This is exactly the metric calculated inside eeg_quality_check().
score = np.sum(detector_psd[line_bins])
line_flags = line_metric > THRESHOLDS[0]
visible_frequencies = freq <= 60

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(freq[visible_frequencies], np.maximum(raw_psd[visible_frequencies], 1e-12),
                 color="#94a3b8", label="Raw")
axes[0].semilogy(freq[visible_frequencies], np.maximum(detector_psd[visible_frequencies], 1e-12),
                 color="#2563eb", label="Detector input")
axes[0].axvspan(49, 51, alpha=0.2, color="#f59e0b", label="49–51 Hz bins")
axes[0].set(xlim=(0, 60), xlabel="Frequency (Hz)", ylabel=f"PSD ({UNIT}²/Hz)",
            title=f"Second {example_second}: bin sum = {score:.2f}")
axes[0].legend(fontsize=8)
axes[1].plot(seconds, line_metric[ch], color="#2563eb", lw=1)
axes[1].axhline(THRESHOLDS[0], color="#dc2626", ls="--", label="Threshold = 10")
axes[1].scatter(seconds[line_flags[ch]], line_metric[ch, line_flags[ch]],
                s=16, color="#dc2626", label="Flagged window")
axes[1].set_yscale("symlog", linthresh=THRESHOLDS[0])
axes[1].set(xlabel="Window start (nominal s)", ylabel=f"PSD-bin sum ({UNIT}²/Hz)",
            title=f"{int(line_flags[ch].sum())}/{n_seconds} windows flagged")
axes[1].legend(fontsize=8)
fig.suptitle(f"2. {CHANNEL_NAMES[ch]}: 50 Hz screening" + REFERENCE_ONLY)
show_and_save(fig, "02-line-noise.png")


### B. High-frequency / possible muscle-activity screening

**Rule:** sum the filtered PSD bins from **20 through 40 Hz**; flag a window
if the result is **greater than 20**. The example below is the selected
channel's largest high-frequency-metric window, not necessarily the same
window as panel A.

Muscle activity can raise high-frequency energy, but this band also contains
brain activity and other noise. Call it an **EMG candidate / high-frequency
flag**, not confirmed muscle contamination. With only EEG, the metric does
not identify the physiological source.


In [ ]:
example_second = int(np.argmax(high_frequency_metric[ch]))
window = slice(example_second * FS, (example_second + 1) * FS)
freq, raw_psd = welch(data[ch, window], fs=FS, nperseg=FS)
_, detector_psd = welch(filtered[ch, window], fs=FS, nperseg=FS)
high_frequency_bins = (freq >= 20.0) & (freq <= 40.0)
score = np.sum(detector_psd[high_frequency_bins])
high_frequency_flags = high_frequency_metric > THRESHOLDS[1]
visible_frequencies = freq <= 60

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(freq[visible_frequencies], np.maximum(raw_psd[visible_frequencies], 1e-12),
                 color="#94a3b8", label="Raw")
axes[0].semilogy(freq[visible_frequencies], np.maximum(detector_psd[visible_frequencies], 1e-12),
                 color="#2563eb", label="Detector input")
axes[0].axvspan(20, 40, alpha=0.16, color="#f59e0b", label="20–40 Hz bins")
axes[0].set(xlim=(0, 60), xlabel="Frequency (Hz)", ylabel=f"PSD ({UNIT}²/Hz)",
            title=f"Second {example_second}: bin sum = {score:.2f}")
axes[0].legend(fontsize=8)
axes[1].plot(seconds, high_frequency_metric[ch], color="#2563eb", lw=1)
axes[1].axhline(THRESHOLDS[1], color="#dc2626", ls="--", label="Threshold = 20")
axes[1].scatter(seconds[high_frequency_flags[ch]], high_frequency_metric[ch, high_frequency_flags[ch]],
                s=16, color="#dc2626", label="Flagged window")
axes[1].set_yscale("symlog", linthresh=THRESHOLDS[1])
axes[1].set(xlabel="Window start (nominal s)", ylabel=f"PSD-bin sum ({UNIT}²/Hz)",
            title=f"{int(high_frequency_flags[ch].sum())}/{n_seconds} windows flagged")
axes[1].legend(fontsize=8)
fig.suptitle(f"3. {CHANNEL_NAMES[ch]}: high-frequency screening, not proof of EMG" + REFERENCE_ONLY)
show_and_save(fig, "03-high-frequency.png")


### C. Large-amplitude sample screening

**Rule:** in each second, count filtered samples with **|amplitude| ≥ 100 µV**;
flag the window when the count is **greater than 2** (at least three samples).
The count is not the number of blinks, movements, or independent events.

The plot uses the selected channel's largest-count window. Red points mark
samples counted by the rule. A transient can spread after filtering; these
points alone cannot distinguish eye movement, electrode motion, or another
cause. Inspect the raw trace and acquisition notes before naming an artifact.


In [ ]:
example_second = int(np.argmax(outlier_count[ch]))
window = slice(example_second * FS, (example_second + 1) * FS)
segment = filtered[ch, window]
amplitude_limit = PROFILE.quality.outlier_absolute_amplitude
extreme_samples = (segment <= -amplitude_limit) | (segment >= amplitude_limit)
count = np.sum(extreme_samples)
outlier_flags = outlier_count > THRESHOLDS[2]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(time_s[window], segment, color="#2563eb", lw=1)
axes[0].scatter(time_s[window][extreme_samples], segment[extreme_samples],
                color="#dc2626", s=12, label="Counted sample", zorder=3)
for limit in (-amplitude_limit, amplitude_limit):
    axes[0].axhline(limit, color="#dc2626", ls="--", lw=0.8)
axes[0].set(xlabel="Nominal time (s)", ylabel=f"Filtered amplitude ({UNIT})",
            title=f"Second {example_second}: {int(count)} extreme samples")
axes[0].legend(fontsize=8)
axes[1].plot(seconds, outlier_count[ch], color="#2563eb", lw=1)
axes[1].axhline(THRESHOLDS[2], color="#dc2626", ls="--", label="Count threshold = 2")
axes[1].scatter(seconds[outlier_flags[ch]], outlier_count[ch, outlier_flags[ch]],
                s=16, color="#dc2626", label="Flagged window")
axes[1].set(xlabel="Window start (nominal s)", ylabel="Extreme samples per 1 s",
            title=f"{int(outlier_flags[ch].sum())}/{n_seconds} windows flagged")
axes[1].legend(fontsize=8)
fig.suptitle(f"4. {CHANNEL_NAMES[ch]}: amplitude screening" + REFERENCE_ONLY)
show_and_save(fig, "04-amplitude-outliers.png")


## 4. Combine the checks into a gate · 15 min

For each channel and second:

`issue = line_flag OR high_frequency_flag OR amplitude_flag`

A channel with issues in **more than 40%** of full seconds becomes a
**bad-channel candidate**. The original gate excludes these channels from
voting, then rejects any second flagged by any remaining channel. Report
channel availability and accepted duration separately: excluding more
channels can make temporal retention look better without improving the EEG.

The original function below returns a compacted array. We show it for source
transparency, but **never use that concatenated array as a continuous signal**.
Instead, `gate_from_metrics()` preserves sample positions and explicitly fixes:

- **All channels bad:** no usable channel means no accepted samples.
- **An incomplete final second:** unassessed, not silently accepted.

For new files with unknown units, the wrapper withholds acceptance. Candidate
channels are also excluded from the channel × sample acceptance mask. The
remaining samples have *passed these heuristics*, not been proven artifact-free.
In the final waveform panel, an excluded viewing channel is replaced by the
first eligible channel, if any, so you can see accepted blocks and rejected
gaps. The displayed channel is named explicitly; this changes no decisions.


In [ ]:
def clean_eeg_data(
    eeg_data: np.ndarray,
    metrics: Tuple[np.ndarray, np.ndarray, np.ndarray],
    thresholds: Tuple[float, float, float],
    segment_length: int = PROFILE.sampling_rate_hz,
    bad_channel_ratio: float = PROFILE.quality.bad_channel_segment_ratio,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, object]]:
    """Reject noisy segments while excluding globally bad channels from voting."""

    matrix = np.asarray(eeg_data, dtype=float)
    if matrix.ndim != 2:
        raise ValueError("clean_eeg_data expects shape (channels, samples).")

    n_segments = metrics[0].shape[1]
    line_bad = metrics[0] > thresholds[0]
    emg_bad = metrics[1] > thresholds[1]
    outlier_bad = metrics[2] > thresholds[2]
    issue_mask = line_bad | emg_bad | outlier_bad
    bad_ratios = np.mean(issue_mask, axis=1) if n_segments else np.zeros(matrix.shape[0])
    bad_indices = np.where(bad_ratios > bad_channel_ratio)[0]
    good_indices = np.where(bad_ratios <= bad_channel_ratio)[0]

    if len(good_indices):
        rejected_segments = np.any(issue_mask[good_indices], axis=0)
    else:
        rejected_segments = np.zeros(n_segments, dtype=bool)

    rejected_points = np.repeat(rejected_segments, segment_length)
    if len(rejected_points) < matrix.shape[1]:
        rejected_points = np.concatenate(
            [
                rejected_points,
                np.zeros(matrix.shape[1] - len(rejected_points), dtype=bool),
            ]
        )
    else:
        rejected_points = rejected_points[: matrix.shape[1]]

    keep_mask = ~rejected_points
    clean = matrix[:, keep_mask]
    channel_names = list(PROFILE.channels)
    info = {
        "bad_channels": bad_indices.tolist(),
        "bad_channel_names": [channel_names[index] for index in bad_indices],
        "channel_bad_ratios": {
            channel_names[index]: float(bad_ratios[index])
            for index in range(matrix.shape[0])
        },
        "retention_rate": float(clean.shape[1] / max(matrix.shape[1], 1)),
        "rejected_segments_count": int(np.sum(rejected_segments)),
        "rejected_segment_indices": np.where(rejected_segments)[0].tolist(),
        "thresholds": {
            "power_50hz": thresholds[0],
            "emg_power": thresholds[1],
            "outlier_count": thresholds[2],
        },
    }
    return clean, keep_mask, info


In [ ]:
# Original behavior is a reference only; do not analyze its compacted timeline.
upstream_compacted, upstream_keep, upstream_info = clean_eeg_data(
    filtered, metrics, THRESHOLDS, segment_length=FS)
del upstream_compacted  # Keep the original data and sample indices instead.

gate = gate_from_metrics(data, metrics, fs=FS, unit_confirmed=unit_confirmed,
                         thresholds=THRESHOLDS)
issue_mask = line_flags | high_frequency_flags | outlier_flags
assert np.array_equal(issue_mask, gate["issue_mask"])
bad_channel_mask = gate["bad_channel_mask"]
sample_keep_mask = gate["keep_mask"]
accepted_channel_samples = (~bad_channel_mask[:, None]) & sample_keep_mask[None, :]
candidate_names = [name for name, bad in zip(CHANNEL_NAMES, bad_channel_mask) if bad]
usable_channel_names = [name for name, bad in zip(CHANNEL_NAMES, bad_channel_mask) if not bad]
accepted_duration_s = float(sample_keep_mask.sum() / FS) if unit_confirmed else None
assert sample_keep_mask.shape == (n_samples,)
assert not sample_keep_mask[n_seconds * FS:].any()
print("Gate status:", gate["summary"]["status"])
print("Bad-channel candidates:", candidate_names or "none", REFERENCE_ONLY)
print("Channels eligible to vote:", usable_channel_names or "none", REFERENCE_ONLY)
print("Accepted nominal duration:", accepted_duration_s, "s (None = not evaluated)")
print("Trailing duration not assessed:", tail_samples / FS, "s")
print("Safeguard-changed sample decisions vs upstream:", int(np.sum(upstream_keep != sample_keep_mask)))
print("Excluded channel names are not inferred scalp locations. The gate does not repair EEG.")


In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(3, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [2, 0.6, 2]})
axes[0].imshow(issue_mask, origin="lower", aspect="auto", interpolation="nearest",
                cmap=ListedColormap(["#e2e8f0", "#dc2626"]), vmin=0, vmax=1,
                extent=(0, n_seconds, -0.5, n_channels - 0.5))
axes[0].set_yticks(np.arange(n_channels),
                   [name + (" *" if bad else "") for name, bad in zip(CHANNEL_NAMES, bad_channel_mask)])
axes[0].set(xlim=(0, n_samples / FS), title="Any of the three checks: red = flag; * = bad-channel candidate")
axes[0].set_xlabel("Window position (nominal s)")

# A separate temporal gate: 0 rejected, 1 accepted, 2 unassessed, 3 units unconfirmed.
state = np.full(n_samples, 2, dtype=int)
state[:n_seconds * FS] = 0 if unit_confirmed else 3
state[sample_keep_mask] = 1
axes[1].imshow(state[None, :], aspect="auto", interpolation="nearest", vmin=0, vmax=3,
                cmap=ListedColormap(["#dc2626", "#16a34a", "#94a3b8", "#a855f7"]),
                extent=(0, n_samples / FS, 0, 1))
axes[1].set_yticks([])
axes[1].set(xlabel="Original nominal time (s)", title="Temporal gate on eligible channels only")
axes[1].legend(handles=[Patch(color=c, label=label) for c, label in
                        [("#dc2626", "Rejected"), ("#16a34a", "Accepted"),
                         ("#94a3b8", "Unassessed tail"), ("#a855f7", "Units unconfirmed")]],
                loc="upper center", bbox_to_anchor=(0.5, -0.7), ncol=4, fontsize=8)

eligible_indices = np.flatnonzero(~bad_channel_mask)
gate_view_channel = (int(eligible_indices[0])
                     if bad_channel_mask[ch] and len(eligible_indices) else ch)
axes[2].plot(time_s, data[gate_view_channel], color="#94a3b8", lw=0.55, label="Original raw signal")
axes[2].plot(time_s, np.where(accepted_channel_samples[gate_view_channel], data[gate_view_channel], np.nan),
             color="#16a34a", lw=0.8, label="Accepted samples on this channel")
if tail_samples:
    axes[2].axvspan(n_seconds, n_samples / FS, color="#94a3b8", alpha=0.3)
shown_duration = "not evaluated" if accepted_duration_s is None else f"{accepted_duration_s:.1f} s"
axes[2].set(xlim=(0, n_samples / FS), xlabel="Original nominal time (s; no concatenation)",
            ylabel=f"Amplitude ({UNIT})",
            title=f"{CHANNEL_NAMES[gate_view_channel]} — candidate excluded: {bool(bad_channel_mask[gate_view_channel])}; temporal acceptance: {shown_duration}")
axes[2].legend(fontsize=8)
fig.suptitle("5. Quality gate: keep the timeline; report channels and duration separately" + REFERENCE_ONLY)
show_and_save(fig, "05-quality-gate.png")


## 5. One small experiment, without changing the detector · 10 min

**Question:** how do flag rates change when each metric's threshold is
halved or doubled? Keep the real signal, filter, 1-second windows, and
metric calculations unchanged. The 100 µV amplitude limit also stays fixed;
for the third detector, only the **count threshold** changes.

The plot compares the fraction of **channel × second** observations flagged
by each detector. These are flag rates, not accuracy, accepted duration,
or evidence that a more permissive threshold is better. We do not rerun
bad-channel selection to optimize retention. The final gate above and saved
summary retain the original thresholds `(10, 20, 2)`.


In [ ]:
threshold_scales = np.array([0.5, 1.0, 2.0])
metric_names = ["49–51 Hz bin sum", "20–40 Hz bin sum", "Extreme-sample count"]
flag_fractions = np.array([
    [np.mean(metric > threshold * scale) for scale in threshold_scales]
    for metric, threshold in zip(metrics, THRESHOLDS)
])
assert np.all(np.diff(flag_fractions, axis=1) <= 0)
fig, ax = plt.subplots(figsize=(10, 4.5))
positions = np.arange(3)
for i, (name, color) in enumerate(zip(metric_names, ["#2563eb", "#f59e0b", "#7c3aed"])):
    bars = ax.bar(positions + (i - 1) * 0.24, 100 * flag_fractions[i], width=0.24,
                  color=color, label=name)
    ax.bar_label(bars, fmt="%.1f%%", fontsize=8, padding=3)
ax.set_xticks(positions, ["0.5 × original threshold", "Original threshold", "2 × original threshold"])
ax.set(ylim=(0, 115), ylabel="Flagged channel-seconds (%)",
       title="6. Threshold sensitivity on the same real data" + REFERENCE_ONLY)
ax.legend(loc="upper right", fontsize=8)
show_and_save(fig, "06-threshold-experiment.png")


In [ ]:
import scipy
import matplotlib

summary = {
    "lesson": 3, "mode": "replay", "synthetic": False,
    "source_sha256": SOURCE_SHA256,
    "source_provenance": "user-supplied real recording; no rescaling or injected artifacts",
    "upstream_commit": "e3539cbeca99e824dfeb3ebdb44d80ca80f0d5dc",
    "amplitude_unit": "uV" if unit_confirmed else "unconfirmed",
    "unit_confirmed": unit_confirmed,
    "unit_confirmation_basis": (
        "owner confirmation on 2026-09-21; matched file SHA-256"
        if SOURCE_SHA256 in CONFIRMED_UV_HASHES and UNIT_CONFIRMED is None
        else "explicit notebook setting" if UNIT_CONFIRMED is not None else "not confirmed"),
    "sample_rate_hz": FS, "time_axis": "nominal sample index / FS; original positions retained",
    "channel_mapping": "unconfirmed; column labels only",
    "channel_labels": CHANNEL_NAMES,
    "shape_channels_by_samples": list(data.shape),
    "malformed_row_count": len(info["malformed_rows"]),
    "packet_clock_summary": timing, "p_field_summary": p_summary,
    "experimental_events": "not inferred; requires a documented event source",
    "preprocessing": "upstream 4th-order Butterworth 1–50 Hz bandpass + filtfilt; no notch",
    "metric_definitions": ["49–51 Hz PSD-bin sum", "20–40 Hz PSD-bin sum", "count abs(filtered) >= 100"],
    "metric_units": [f"{UNIT}^2/Hz", f"{UNIT}^2/Hz", "samples per full second"],
    "quality_gate": gate["summary"],
    "accepted_duration_s": accepted_duration_s,
    "candidate_channel_names": candidate_names,
    "eligible_channel_names": usable_channel_names,
    "unassessed_tail_samples": tail_samples,
    "threshold_experiment": {"scales": threshold_scales.tolist(), "flag_fractions": flag_fractions.tolist(),
                             "changes_final_gate": False},
    "figures": FIGURES,
    "software": {"python": sys.version.split()[0], "numpy": np.__version__,
                 "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "limitations": ["heuristic screening, not artifact diagnosis or repair",
                    "offline replay only; live-device validation not performed",
                    "nominal time does not repair irregular device clocks"],
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
np.savez_compressed(OUTPUT_DIR / "quality-masks.npz", keep_mask=sample_keep_mask,
                    assessed_mask=gate["assessed_mask"], bad_channel_mask=bad_channel_mask,
                    accepted_channel_samples=accepted_channel_samples, issue_mask=issue_mask)
print("Saved six figures, summary.json, and quality-masks.npz under:")
print(OUTPUT_DIR.relative_to(ROOT))
print("These local outputs are ignored by Git. Original recordings were not changed.")


## Check your understanding

1. Which three quantities did we measure? What does a red point mean?
2. Why can we not call every high-frequency flag a muscle artifact?
3. Why can a high temporal retention rate hide the loss of important channels?
4. What should the gate do if every channel is bad, or the last second is incomplete?
5. Why must rejected gaps remain visible before downstream spectral analysis?

**Deliverable:** select one real window and describe its three scores, the
resulting flag(s), and one limitation. If there are no flags, report that;
do not fabricate an artifact. Include channel availability, accepted duration,
and unassessed trailing duration. A clean-looking trace is not a QC certificate.

**Next:** Lesson 4 develops filtering and PSD. Do not simply concatenate
accepted windows and treat the result as a continuous recording; analyze
valid continuous blocks and preserve any experimental boundaries.

**Source and reproducibility:** see `docs/lesson03-source.md` for the pinned
upstream source, MIT notice, exact threshold semantics, and wrapper changes.
No live device connection, clinical conclusion, or raw-data publication is
part of this lesson.
